In [1]:
from datetime import datetime

import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.set_printoptions(suppress=True, precision=5) # Disable scientific notation

SELECTED_RUNS = [72, 73]

PACEJKA_PARAMS_NAMES = ["B", "C", "D", "E", "F"]
PACEJKA_PARAMS_GUESS = [-18, -0.01, 100000, -2, 0.00076]  # Update as needed

def quad_fit(Fz, a0, a1, a2):
    return a0 + a1 * Fz + a2 * (Fz ** 2)

def lin_fit(Fz, a0, a1):
    return a0 + Fz * a1

def exp_fit(Fz, a0, a1, a2):
    return a0 + a1 * (np.e ** (a2 * Fz))

PACEJKA_PARAMS_FIT_FNS = [quad_fit, quad_fit, lin_fit, exp_fit, quad_fit]
PACEJKA_PARAMS_FIT_GUESS = [
    [-0.37, -0.00045, -0.0000002],
    [0, 0, 0],
    [0, 0],
    [0, 0, 0],
    [0, 0, 0]
]

In [2]:
def get_run_data(selected_runs):
    metric_data = pd.DataFrame()

    for rn in selected_runs:
        # Import the data
        metric_datum = pd.read_csv(f'./SN5_R9_Longitudinal/run{rn}.csv')

        metric_datum["ET"] = metric_datum["ET"] - metric_datum["ET"].iloc[0]
        
        if len(metric_data) != 0:
            metric_datum["ET"] = metric_datum["ET"] + metric_data["ET"].iloc[-1]

        metric_data = pd.concat([metric_data, metric_datum])
        print(f"Loaded file run{rn}.dat ({len(metric_datum)} rows)")

    return metric_data

def bin_by_param(df, bins, param_col):
    out = []
    for lb, ub in bins:
        sub = df[(df[param_col] >= lb) & (df[param_col] < ub)]
        out.append(sub)
    return out

In [3]:
def pacejka_longitudinal(sl, B, C, D, E, F):
    return D * np.sin(C * np.arctan(B * sl - E * (B * sl - np.arctan(B * sl))) + F)

def pacejka_longitudinal_with_parameter(sl, Fz, hyper_coefficients):
    B, C, D, E, F = [PACEJKA_PARAMS_FIT_FNS[i](Fz, *hyper_coefficients[i]) for i in range(5)]

    return D * np.sin(C * np.arctan(B * sl - E * (B * sl - np.arctan(B * sl))) + F)

# def pacejka_lateral_derivative_with_parameter(alpha, Fz, hyper_coefficients):
#         B, C, D, E, F = [PACEJKA_PARAM_FIT_FNS[i](Fz, *hyper_coefficients[i]) for i in range(5)]

#         return -(C * D * (E * (B - B / ((B ** 2) * (alpha ** 2) + 1)) - B) * np.cos(C * np.arctan(E * (B * alpha - np.arctan(B * alpha)) - B * alpha) - F)) / (((E * (B * alpha - np.arctan(B * alpha)) - B * alpha) ** 2) + 1)

def fit_pacejka(df, x_col, y_col):
    popt, _ = curve_fit(
        pacejka_longitudinal, 
        df[x_col], 
        df[y_col], 
        p0=PACEJKA_PARAMS_GUESS, 
        maxfev=50000
    )

    return popt

In [4]:
all_data = get_run_data(SELECTED_RUNS)

Loaded file run72.dat (75860 rows)
Loaded file run73.dat (66778 rows)


In [5]:
Fz_bins = [(-2000, -900), (-900, -725), (-725, -400), (-400, -250), (-250, 0)] # Empirically found by looking at graph.
all_data_binned = bin_by_param(all_data, Fz_bins, "FZ")

params_binned = np.zeros((len(Fz_bins), 5))
for i in range(5):
    data_bin = all_data_binned[i]
    params_binned[i, :] = fit_pacejka(data_bin, "SL", "FX")
    print(f"Fitted bin {i + 1}")

Fitted bin 1
Fitted bin 2
Fitted bin 3
Fitted bin 4
Fitted bin 5


In [6]:
def surface_fig(all_data, hyper_coefficients, Fz_bins):

    Fz_domain = np.linspace(all_data["FZ"].min(), -225, 100)
    SL_domain = np.linspace(all_data["SL"].min(), all_data["SL"].max(), 100)

    Fy_surface = np.zeros(shape=(len(Fz_domain), len(SL_domain)))
    for i in range(len(Fz_domain)):
        for j in range(len(SL_domain)):
            Fy_surface[j][i] = pacejka_longitudinal_with_parameter(SL_domain[j], Fz_domain[i], hyper_coefficients)

    fig = go.Figure(data=[go.Surface(
        x=Fz_domain,
        y=SL_domain,
        z=Fy_surface
    )])

    # fig.add_trace(go.Scatter3d(
    #     x=[load_force], 
    #     y=[slip_ratio],
    #     z=[pacejka_longitudinal_with_parameter(slip_ratio, load_force, hyper_coefficients)],
    #     marker=dict(
    #         color="white"
    #     )
    # ))

    CA_YELLOW = "#FDB515"
    BK_BLUE = "#002676"

    LABEL_COLOR = BK_BLUE
    TICK_COLOR = CA_YELLOW
    
    SCENE_BACKGROUND = "white" #"Greenscreen", to photoshop out background
    PLANE_COLOR = "#242424"

    fig.update_layout(
        scene=dict(
            bgcolor=SCENE_BACKGROUND,
            xaxis=dict(
                gridcolor=TICK_COLOR,
                title=dict(
                    text='Load Force (N)', 
                    font=dict(
                        color=LABEL_COLOR, 
                        size=20
                    )
                ),
                tickfont=dict(color=LABEL_COLOR),
                showbackground=True,
                backgroundcolor=PLANE_COLOR
            ),
            yaxis=dict(
                gridcolor=TICK_COLOR,
                title=dict(
                    text='Slip Ratio', 
                    font=dict(
                        color=LABEL_COLOR, 
                        size=20
                    )
                ),
                tickfont=dict(color=LABEL_COLOR),
                showbackground=True,
                backgroundcolor=PLANE_COLOR
            ),
            zaxis=dict(
                gridcolor=TICK_COLOR,
                title=dict(
                    text='Fx (N)', 
                    font=dict(
                        color=LABEL_COLOR, 
                        size=20
                    )
                ),
                tickfont=dict(color=LABEL_COLOR),
                showbackground=True,
                backgroundcolor=PLANE_COLOR
            ),
        ),
        # title=dict(
        #     x=0.5,
        #     xanchor='center',
        #     text='Instantaneous Cornering Stiffness',
        #     font=dict(
        #         size=36,
        #     )
        # )
    )
    
    return fig

In [7]:
timestamp = datetime.now().strftime("%m%d_%H%M%S")

hyper_coefficients = []

for i in range(5):
    values_for_this_param = params_binned[:,i] # ith column
    Fz_medians = [(ub + lb) / 2 for lb, ub in Fz_bins]

    coefficients, _ = curve_fit(PACEJKA_PARAMS_FIT_FNS[i], Fz_medians, values_for_this_param, maxfev=150000, p0=PACEJKA_PARAMS_FIT_GUESS[i])
    hyper_coefficients.append(coefficients)
    
surf_fig = surface_fig(all_data, hyper_coefficients, Fz_bins)

surf_fig.write_html(f"fx_out/figures/fx_sens_surf_({timestamp}).html")

In [8]:
def bin_figs(data_binned, params_binned, Fz_bins):
    fig = make_subplots(rows=2, cols=3, subplot_titles=[f"Fx over SL for Fz in bin [{lb:.2f}, {ub:.2f})" for lb, ub in Fz_bins])

    for i in range(5):
        data_bin = data_binned[i]
        graph_domain = np.linspace(min(data_bin["SL"]), max(data_bin["SL"]), 500)

        # Draw scatterplot of data
        fig.add_trace(
            go.Scatter( x=data_bin["SL"], 
                        y=data_bin["FX"], 
                        mode="markers", 
                        # marker=dict(
                        #     color=data_bin["FZ N"], 
                        #     colorscale="Viridis", 
                        #     showscale=True, 
                        #     colorbar=dict(title=f"Bin {i + 1}")
                        # ),
                        name=f'Data for bin {i}',
            ), 
            row=i // 3 + 1, 
            col=i % 3 + 1
        )

        # Draw pacejka curve
        fig.add_trace(
            go.Scatter(
                x=graph_domain, 
                y=pacejka_longitudinal(graph_domain, *params_binned[i]), 
                mode="lines", 
                marker=dict(color="yellow"), 
                name=f'Fitted curve for bin {i}'
            ), 
            row=i // 3 + 1, 
            col=i % 3 + 1
        )
    
    return fig

In [9]:
bin_fig = bin_figs(all_data_binned, params_binned, Fz_bins)
print(hyper_coefficients)
bin_fig.write_html("fx_out/figures/fx_bin_figs.html")

[array([-2.63879271e+01, -1.52489684e-02, -6.58652522e-06]), array([-4.28901613e-04,  1.68030721e-05,  6.86050026e-09]), array([ 7.06280281e+04, -5.35472378e+01]), array([-8.75220232e+00,  7.41226064e+00,  2.49982430e-04]), array([ 2.65927655e-05, -8.47778857e-07, -3.53815907e-10])]


In [10]:
print(hyper_coefficients)
for i in range(5):
    if PACEJKA_PARAMS_FIT_FNS[i] == lin_fit:
        print(f"{PACEJKA_PARAMS_NAMES[i]} = {hyper_coefficients[i][0]} + {hyper_coefficients[i][1]} * Fz")
    elif PACEJKA_PARAMS_FIT_FNS[i] == quad_fit:
        print(f"{PACEJKA_PARAMS_NAMES[i]} = {hyper_coefficients[i][0]} + {hyper_coefficients[i][1]} * Fz + {hyper_coefficients[i][2]} * (Fz ** 2)")
    elif PACEJKA_PARAMS_FIT_FNS[i] == exp_fit:
        print(f"{PACEJKA_PARAMS_NAMES[i]} = {hyper_coefficients[i][0]} + {hyper_coefficients[i][1]} * (Math.E ** ({hyper_coefficients[i][2]} * Fz))")
        

[array([-2.63879271e+01, -1.52489684e-02, -6.58652522e-06]), array([-4.28901613e-04,  1.68030721e-05,  6.86050026e-09]), array([ 7.06280281e+04, -5.35472378e+01]), array([-8.75220232e+00,  7.41226064e+00,  2.49982430e-04]), array([ 2.65927655e-05, -8.47778857e-07, -3.53815907e-10])]
B = -26.38792705599439 + -0.015248968445049783 * Fz + -6.586525215887752e-06 * (Fz ** 2)
C = -0.000428901612800267 + 1.6803072084681463e-05 * Fz + 6.860500259858938e-09 * (Fz ** 2)
D = 70628.02807298147 + -53.54723776768741 * Fz
E = -8.75220232051474 + 7.4122606400675926 * (Math.E ** (0.00024998243011583606 * Fz))
F = 2.6592765480231e-05 + -8.477788573473665e-07 * Fz + -3.538159069988727e-10 * (Fz ** 2)


In [11]:
from dash import Dash, html, dcc, callback, Output, Input

app = Dash()

initial_slip = 0
initial_load = -800

app.layout = html.Div(children=[
    dcc.Graph(
        id='surface_figure',
        figure=surf_fig,
        style={'width': '100vw', 'height': '100vh', 'position': 'absolute', 'top': 0, 'left': 0, 'z-index': 0}
    ),
    
    html.Div([ "Slip angle (deg): ",
            dcc.Input(
                id='slip-angle', 
                value=0, 
                type='number'
            )
        ],
        style={'position': 'absolute', 'top': "5vh", 'left': "70vw", 'z-index': 100}
    ),

    html.Div([ "Load Force (N): ",
            dcc.Input(
                id='load-force', 
                value=-800, 
                type='number',
            )
        ],
        style={'position': 'absolute', 'top': "9vh", 'left': "70vw", 'z-index': 100}
    ),

    html.Div([ 
        "Lateral Force (N): ", 
        html.Span(id="current-lateral", children=pacejka_lateral_with_parameter(initial_slip, initial_load, hyper_coefficients))
        ],
        style={'position': 'absolute', 'top': "13vh", 'left': "70vw", 'z-index': 100}
    ),
])

@callback(
    Output('surface_figure', 'figure'),
    Output('current-lateral', 'children'),
    Input(component_id='slip-angle', component_property='value'),
    Input(component_id='load-force', component_property='value')
)
def update_output_div(slip_angle, load_force):
    return surface_fig(all_data, hyper_coefficients, Fz_bins, slip_angle, load_force), pacejka_lateral_with_parameter(slip_angle, load_force, hyper_coefficients)

app.run(jupyter_mode="external", debug=True)

NameError: name 'pacejka_lateral_with_parameter' is not defined